# GOES ABI-L2-MCMIPC preview

Interactive preview of the downloaded GOES imagery in `data/goes/`. Pick a date,
view a band (or stack three into an RGB), and explore at the native
**2 km / 2500×1500** CONUS resolution.

Every file holds all 16 ABI bands:

| # | µm | type | # | µm | type |
|---|----|------|---|----|------|
| 1 | 0.47 blue | reflectance | 9 | 6.9 mid water-vapor | brightness temp |
| 2 | 0.64 red | reflectance | 10 | 7.3 low water-vapor | brightness temp |
| 3 | 0.86 veggie | reflectance | 11 | 8.4 cloud-top phase | brightness temp |
| 4 | 1.37 cirrus | reflectance | 12 | 9.6 ozone | brightness temp |
| 5 | 1.6 snow/ice | reflectance | 13 | 10.3 clean IR | brightness temp |
| 6 | 2.2 cloud particle | reflectance | 14 | 11.2 IR | brightness temp |
| 7 | 3.9 shortwave IR | brightness temp | 15 | 12.3 dirty IR | brightness temp |
| 8 | 6.2 upper water-vapor | brightness temp | 16 | 13.3 CO₂ | brightness temp |

True color ≈ RGB from bands **(2, 3, 1)**.

> **Zoom:** the next cell enables `%matplotlib widget`, so every figure is
> interactive — scroll to zoom, drag to pan, down to single pixels. For a
> targeted view, `crop_lonlat(...)` / `crop_pixels(...)` subset first (still full res).

In [ ]:
# Interactive zoom/pan on every figure (scroll = zoom, drag = pan).
# If the widget backend isn't available, switch this to:  %matplotlib inline
%matplotlib widget

## Helpers — run this cell once

In [ ]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyproj
import xarray as xr

# data/goes whether the notebook runs from notebooks/ or the repo root
DATA_DIR = next(
    (p for p in [Path("../data/goes"), Path("data/goes")] if p.exists()),
    Path("../data/goes"),
)


# ---- file discovery & selection ----
def _scan_token(p):
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def scan_time(p):
    t = _scan_token(p)
    if t.startswith("s") and len(t) >= 12:
        return f"{t[8:10]}:{t[10:12]} UTC"
    return "??:??"


def find_files(dt, data_dir=DATA_DIR):
    pat = f"*/{dt.year}/{dt.month:02d}/{dt.day:02d}/*.nc"
    return sorted(data_dir.glob(pat), key=_scan_token)


def open_goes(dt, file_index=0, data_dir=DATA_DIR):
    """List the files for a date and open one (default the first/only)."""
    files = find_files(dt, data_dir)
    if not files:
        raise FileNotFoundError(f"No .nc files for {dt} under {data_dir}")
    for i, p in enumerate(files):
        arrow = "->" if i == file_index else "  "
        print(f"{arrow} [{i}] {scan_time(p)}  {p.name}")
    return xr.open_dataset(files[file_index], decode_times=False)


# ---- band access & scaling ----
def band(ds, n):
    return ds[f"CMI_C{n:02d}"]


def _is_bt(da):
    return str(da.attrs.get("units", "")).strip().upper().startswith("K")


def _cmap(da):
    return "gray_r" if _is_bt(da) else "gray"  # IR: cold cloud tops = white


def _stretch(a, vmin=None, vmax=None):
    lo = np.nanpercentile(a, 2) if vmin is None else vmin
    hi = np.nanpercentile(a, 98) if vmax is None else vmax
    return float(lo), float(hi)


def _norm(a, lo, hi):
    return np.clip((a - lo) / (hi - lo), 0, 1) if hi > lo else np.zeros_like(a)


# ---- geolocation & cropping (native resolution preserved) ----
def _geos(ds):
    return pyproj.CRS.from_cf(dict(ds["goes_imager_projection"].attrs))


def crop_lonlat(ds, lon_min, lon_max, lat_min, lat_max):
    """Subset to a lon/lat box (degrees); still renders at full 2 km resolution."""
    h = ds["goes_imager_projection"].attrs["perspective_point_height"]
    tf = pyproj.Transformer.from_crs("EPSG:4326", _geos(ds), always_xy=True)
    lons = [lon_min, lon_max, lon_min, lon_max]
    lats = [lat_min, lat_min, lat_max, lat_max]
    xs, ys = tf.transform(lons, lats)
    xs = np.array(xs) / h
    ys = np.array(ys) / h
    xs, ys = xs[np.isfinite(xs)], ys[np.isfinite(ys)]
    if xs.size == 0 or ys.size == 0:
        raise ValueError("box is off the Earth disk for this satellite")
    return ds.sel(x=slice(xs.min(), xs.max()), y=slice(ys.max(), ys.min()))


def crop_pixels(ds, x0, x1, y0, y1):
    """Subset by pixel index (x: 0..2500 W->E, y: 0..1500 N->S)."""
    return ds.isel(x=slice(x0, x1), y=slice(y0, y1))


# ---- rendering ----
def show_band(ds, n, cmap=None, vmin=None, vmax=None, figsize=(11, 7)):
    da = band(ds, n)
    a = da.values
    lo, hi = _stretch(a, vmin, vmax)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(a, cmap=cmap or _cmap(da), vmin=lo, vmax=hi,
                   interpolation="nearest", origin="upper", aspect="equal")
    units = da.attrs.get("units", "")
    ax.set_title(f"Band {n}: {da.attrs.get('long_name', '')} ({units})\n"
                 f"{a.shape[1]}x{a.shape[0]} px")
    fig.colorbar(im, ax=ax, shrink=0.7, label=units)
    fig.tight_layout()
    return fig


def show_rgb(ds, bands=(2, 3, 1), gamma=2.2, figsize=(11, 7)):
    chans = []
    for b in bands:
        a = band(ds, b).values
        lo, hi = _stretch(a)
        chans.append(_norm(a, lo, hi))
    rgb = np.clip(np.nan_to_num(np.dstack(chans), nan=0.0) ** (1 / gamma), 0, 1)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb, interpolation="nearest", origin="upper", aspect="equal")
    ax.set_title(f"RGB = bands {tuple(bands)} (R, G, B)\n"
                 f"{rgb.shape[1]}x{rgb.shape[0]} px")
    fig.tight_layout()
    return fig

## 1. Choose a date & file

Each downloaded day has one image (18:00 UTC). If a day has several, `open_goes`
lists them — pass `file_index=` to choose.

In [ ]:
DATE = date(2024, 11, 17)
ds = open_goes(DATE, file_index=0)

## 2. Single band

Reflectance bands (1–6) render grayscale; IR bands (7–16) invert so cold cloud
tops appear white.

In [ ]:
show_band(ds, 13);   # 13 = clean IR window. Try 2 (red), 8 (water vapor), ...

## 3. True-color RGB

Stack three bands as R, G, B. Bands `(2, 3, 1)` ≈ natural color (daytime only).

In [ ]:
show_rgb(ds, bands=(2, 3, 1));

## 4. Zoom to a region (native resolution)

Subset **before** plotting to focus on an area — e.g. a flood's lon/lat box — and
it still renders at full 2 km resolution. Combine with the live widget zoom for the
finest detail.

In [ ]:
# Zoom to a lon/lat box (degrees) - e.g. the Texas/Louisiana coast
sub = crop_lonlat(ds, lon_min=-98, lon_max=-92, lat_min=28, lat_max=33)
show_band(sub, 13);

In [ ]:
# Or subset by pixel index (x: 0..2500 W->E, y: 0..1500 N->S)
sub = crop_pixels(ds, x0=1000, x1=1700, y0=550, y1=1050)
show_rgb(sub, bands=(2, 3, 1));

## Tips

- **Change the view:** edit `DATE`, the band in `show_band(ds, N)`, or the RGB triple.
- **Save full-resolution:** `fig = show_band(ds, 13); fig.savefig("out.png", dpi=200)`
- **Live zoom:** with `%matplotlib widget`, scroll to zoom and drag to pan; the home
  button resets the view.
- **Contrast:** `show_band(ds, 13, vmin=200, vmax=300)` (Kelvin for IR; 0–1 reflectance).